# Apriori

Apriori is an **association rule learning algorithm** that finds items frequently occurring together in transactions - for example, customers who buy pasta often also buy tomato sauce.

It evaluates rules using:

- **Support:** How often the items occur together.
- **Confidence:** How often the second item appears when the first is present.
- **Lift:** How much more often they occur together than expected by chance. A lift above 1 suggests a positive association.

It reduces computation by skipping larger item combinations when their smaller subsets are already infrequent.

## Importing the libraries

In [1]:
from pandas import DataFrame
from pandas.io.parsers import TextFileReader
!pip install apyori

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

## Data Preprocessing

In [3]:
dataset = pd.read_csv('data/Market_Basket_Optimisation.csv', header = None)
transactions = dataset.to_numpy(dtype=str).tolist()
# for i in range(0, 7501):
#   transactions.append([str(dataset.values[i,j]) for j in range(0, 20)])

## Training the Apriori model on the dataset

In [4]:
from apyori import apriori
rules = apriori(transactions = transactions, min_support = 0.003, min_confidence = 0.2, min_lift = 3, min_length = 2, max_length = 2)

## Visualising the results

### Displaying the first results coming directly from the output of the apriori function

In [5]:
results = list(rules)

Example record having the rule **light cream → chicken**: baskets containing light cream often also contain chicken.

```
RelationRecord(
    items=frozenset({'light cream', 'chicken'}),
    support=0.004532728969470737,
    ordered_statistics=[
        OrderedStatistic(
            items_base=frozenset({'light cream'}),
            items_add=frozenset({'chicken'}),
            confidence=0.29059829059829057,
            lift=4.84395061728395
        )])
```

| Property | Meaning in this example |
|---|---|
| `items` | The item combination being examined: **light cream and chicken**. |
| `support = 0.00453` | **0.453% of all transactions** contain both items—34 out of 7,501 baskets. |
| `ordered_statistics` | A list of directional rules derived from this combination that meet your configured thresholds. |
| `items_base` | The rule’s starting condition: the basket contains **light cream**. |
| `items_add` | The associated item: **chicken**. |
| `confidence = 0.2906` | **29.06% of baskets containing light cream also contain chicken**. |
| `lift = 4.844` | Chicken is **4.84 times as likely** in baskets containing light cream as it is in baskets overall. |

The relationship between confidence and lift is:


$$
\mathrm{Lift}
= \frac{P(\text{chicken} \mid \text{light cream})}{P(\text{chicken})}
\approx \frac{29.06\%}{6.00\%}
\approx 4.84
$$


`frozenset` simply means an immutable, unordered set of items.

The rule is directional: **chicken → light cream** would have a different confidence. These results show an association; they don’t establish that buying light cream causes someone to buy chicken.

In [6]:
results

[RelationRecord(items=frozenset({'light cream', 'chicken'}), support=0.004532728969470737, ordered_statistics=[OrderedStatistic(items_base=frozenset({'light cream'}), items_add=frozenset({'chicken'}), confidence=0.29059829059829057, lift=4.84395061728395)]),
 RelationRecord(items=frozenset({'mushroom cream sauce', 'escalope'}), support=0.005732568990801226, ordered_statistics=[OrderedStatistic(items_base=frozenset({'mushroom cream sauce'}), items_add=frozenset({'escalope'}), confidence=0.3006993006993007, lift=3.790832696715049)]),
 RelationRecord(items=frozenset({'pasta', 'escalope'}), support=0.005865884548726837, ordered_statistics=[OrderedStatistic(items_base=frozenset({'pasta'}), items_add=frozenset({'escalope'}), confidence=0.3728813559322034, lift=4.700811850163794)]),
 RelationRecord(items=frozenset({'honey', 'fromage blanc'}), support=0.003332888948140248, ordered_statistics=[OrderedStatistic(items_base=frozenset({'fromage blanc'}), items_add=frozenset({'honey'}), confidence=0

### Putting the results well organised into a Pandas DataFrame

In [7]:
def inspect(rules_list):
    lhs         = [tuple(result[2][0][0])[0] for result in rules_list]
    rhs         = [tuple(result[2][0][1])[0] for result in rules_list]
    supports    = [result[1] for result in rules_list]
    confidences = [result[2][0][2] for result in rules_list]
    lifts       = [result[2][0][3] for result in rules_list]
    return list(zip(lhs, rhs, supports, confidences, lifts))

resultsInDataFrame = pd.DataFrame(inspect(results), columns = ['Left Hand Side', 'Right Hand Side', 'Support', 'Confidence', 'Lift'])

### Displaying the results non sorted

In [8]:
resultsInDataFrame

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
0,light cream,chicken,0.004533,0.290598,4.843951
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
2,pasta,escalope,0.005866,0.372881,4.700812
3,fromage blanc,honey,0.003333,0.245098,5.164271
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
5,tomato sauce,ground beef,0.005333,0.377358,3.840659
6,light cream,olive oil,0.003200,0.205128,3.114710
7,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
8,pasta,shrimp,0.005066,0.322034,4.506672


### Displaying the results sorted by descending lifts

In [9]:
resultsInDataFrame.nlargest(n = 10, columns ='Lift')

,Left Hand Side,Right Hand Side,Support,Confidence,Lift
3,fromage blanc,honey,0.003333,0.245098,5.164271
0,light cream,chicken,0.004533,0.290598,4.843951
2,pasta,escalope,0.005866,0.372881,4.700812
8,pasta,shrimp,0.005066,0.322034,4.506672
7,whole wheat pasta,olive oil,0.007999,0.271493,4.122410
5,tomato sauce,ground beef,0.005333,0.377358,3.840659
1,mushroom cream sauce,escalope,0.005733,0.300699,3.790833
4,herb & pepper,ground beef,0.015998,0.323450,3.291994
6,light cream,olive oil,0.003200,0.205128,3.114710
